# Week 19: Time-Series Deep Learning, State Space Models & Mamba
## Notebook 02: Temporal Convolutional Networks (TCN) & Dilated Convolutions

### 1. Conceptual Intuition & Real-World Use Case
For sequential data, RNNs and LSTMs were traditionally dominant. However:
- RNNs compute sequentially step-by-step: step $t$ depends on step $t-1$, which cannot be parallelized across GPU CUDA cores.
- Gradients degrade over long contexts.

In 2018, Bai, Kolter, and Vladu introduced **Temporal Convolutional Networks (TCN)**:
1. **100% Parallel Training:** Convolutions evaluate all sequence positions simultaneously.
2. **Causal Convolutions:** Ensures no information leakage from future to past ($y_t$ depends only on $x_{\le t}$).
3. **Dilated Convolutions:** Exponentially expanding dilation factor ($d = 1, 2, 4, 8, \dots$) allows an **exponentially large receptive field** with linear parameters!

---

### 2. Mathematical Foundations: Dilated Causal Convolution
For a 1D sequence $x \in \mathbb{R}^T$ and a filter $f \in \mathbb{R}^K$ with dilation factor $d$:

$$y_t = (x *_d f)(t) = \sum_{k=0}^{K-1} f_k \cdot x_{t - d \cdot k}$$

#### Receptive Field Growth:
For a TCN with $L$ layers, kernel size $K$, and dilation $d_l = 2^{l-1}$:
$$\text{Receptive Field} = 1 + \sum_{l=1}^L 2 \cdot (K - 1) \cdot d_l = 1 + 2(K - 1)(2^L - 1)$$
With just 8 layers and $K=3$, the network perceives a receptive context of over **1,000 historical time steps**!

In [ ]:
import numpy as np

np.random.seed(42)

# Step 1: Implement 1D Dilated Causal Convolution from Scratch
def dilated_causal_conv1d(x, kernel, dilation):
    """
    x: 1D array of length T
    kernel: 1D filter of length K
    dilation: int d
    """
    T = len(x)
    K = len(kernel)
    output = np.zeros(T)
    
    for t in range(T):
        conv_val = 0.0
        for k in range(K):
            # Causal index stepping backward in time
            past_idx = t - k * dilation
            if past_idx >= 0:
                conv_val += kernel[k] * x[past_idx]
        output[t] = conv_val
    return output

# Step 2: Simulate Multi-Layer TCN with Exponential Dilations (d=1, 2, 4)
T = 16
x_input = np.ones(T) # Constant signal to trace impulse receptive field
kernel = np.array([0.5, 0.5]) # K=2

# Layer 1: d=1 (Receptive field = 2)
h1 = dilated_causal_conv1d(x_input, kernel, dilation=1)

# Layer 2: d=2 (Receptive field = 4)
h2 = dilated_causal_conv1d(h1, kernel, dilation=2)

# Layer 3: d=4 (Receptive field = 8)
h3 = dilated_causal_conv1d(h2, kernel, dilation=4)

print("--- TCN Layer Outputs across Sequence Positions ---")
print("Input (T=16):\n", x_input[:8])
print("\nLayer 1 (d=1):\n", np.round(h1[:8], 3))
print("\nLayer 2 (d=2):\n", np.round(h2[:8], 3))
print("\nLayer 3 (d=4):\n", np.round(h3[:8], 3))

# Calculate theoretical receptive field for K=3, L=6
K = 3
L = 6
receptive_field = 1 + (K - 1) * sum(2**i for i in range(L))
print(f"\nReceptive Field with K={K}, {L} layers: {receptive_field} time steps!")

### 3. Step-by-Step Code Walkthrough
1. **Causal Masking:** Lookups are restricted to indices $t - k \cdot d \le t$, mathematically guaranteeing that future time steps cannot leak backward.
2. **Dilation Strides:** $d=1$ connects adjacent steps, $d=2$ skips 1 step, and $d=4$ skips 3 steps, forming an exponentially growing binary tree across time.
3. **Parallel Compute:** In PyTorch or TensorFlow, this executes as standard 1D convolutional tensor operations across all $T$ positions in parallel on GPU.

---

### 4. Industry Applications & Trade-offs
- **ECG Cardiac Telemetry:** Detecting cardiac arrhythmias across long continuous heart-rate recordings.
- **WaveNet (Google DeepMind):** Used dilated causal convolutions to synthesize human speech audio.

---

### 5. Top Interview Questions & Answers
**Q1: Why do TCNs train substantially faster than LSTMs?**
> **Answer:** LSTMs are autoregressively sequential during training ($h_t$ requires $h_{t-1}$), preventing parallel GPU execution across the time axis. TCNs use convolutions where all time steps are computed simultaneously in parallel via matrix multiplications.

**Q2: What is the purpose of residual connections in deep TCNs?**
> **Answer:** As receptive fields expand by stacking 10+ dilated layers, networks become deep. Residual connections ($x + \text{Conv}(x)$) ensure stable gradient flow directly to early layers, preventing vanishing gradients over long historical context windows.